In [14]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, KFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


warnings.filterwarnings('ignore')

In [8]:
df = pd.read_csv('../data/desafio_nps_fase_1.csv')
df.head()

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9


In [9]:
# Configuração de validação cruzada compartilhada para todos os modelos
shared_cv = KFold(n_splits=10, shuffle=True, random_state=42)

# One Hot Encode a coluna 'customer_region'
one_hot_encoded_regions = pd.get_dummies(df['customer_region'], prefix='region', drop_first=True)

# Features de treinamento
X = df.drop(columns=[
    'customer_id',
    'order_id',
    'customer_region',
    'nps_score'])

# Join das features com as colunas one-hot encoded
X = X.join(one_hot_encoded_regions)
y = df['delivery_delay_days'] # Labels resultado binário

# Divisão dos dados em treino e teste para ambos os conjuntos de labels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [12]:
linear_regression = LinearRegression()
linear_regression.fit(X_train, y_train)
linear_regression_predictions = linear_regression.predict(X_test)

In [15]:
decition_tree = DecisionTreeRegressor(random_state=42)
decition_tree.fit(X_train, y_train)
decition_tree_predictions = decition_tree.predict(X_test)

In [16]:
random_forest = RandomForestRegressor(random_state=42)
random_forest.fit(X_train, y_train)
random_forest_predictions = random_forest.predict(X_test)

In [20]:
print("Linear Regression Predictions:", accuracy_score(y_test, linear_regression_predictions.round()))
print("Decision Tree Predictions:", accuracy_score(y_test, decition_tree_predictions.round()))
print("Random Forest Predictions:", accuracy_score(y_test, random_forest_predictions.round()))

Linear Regression Predictions: 1.0
Decision Tree Predictions: 1.0
Random Forest Predictions: 1.0


In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Carregar o dataset
df = pd.read_csv('../data/desafio_nps_fase_1.csv')

# 2. Pré-processamento
# Remover colunas de identificação (IDs não ajudam a prever atrasos)
colunas_para_remover = ['customer_id', 'order_id', 'nps_score', 'csat_internal_score']  # 'nps_score' é removido pois não é uma feature preditiva
df = df.drop(columns=colunas_para_remover)

df = df[[
    'customer_region',
    'order_value',
    'items_quantity',
    'delivery_time_days',
    'delivery_delay_days',
    'freight_value']]

# Converter a variável categórica 'customer_region' em variáveis dummy (One-Hot Encoding)
df = pd.get_dummies(df, columns=['customer_region'], drop_first=True)

# 3. Separar as Variáveis Preditoras (X) e a Variável Alvo (y)
X = df.drop(columns=['delivery_delay_days'])
y = df['delivery_delay_days']

# 4. Dividir os dados em conjuntos de Treino (80%) e Teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Inicializar e Treinar o Modelo
# Utilizamos 100 árvores de decisão (n_estimators=100)
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train, y_train)

# 6. Fazer previsões com o conjunto de teste
y_pred = modelo_rf.predict(X_test)

# 7. Avaliar a performance do modelo
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== Resultados da Avaliação ===")
print(f"MAE (Erro Médio Absoluto): {mae:.2f} dias")
print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse:.2f} dias")
print(f"R² (Score de Variância): {r2:.2f}\n")

# 8. Extrair as features mais importantes para o atraso
importancias = pd.DataFrame({
    'Variável': X_train.columns, 
    'Importância': modelo_rf.feature_importances_
}).sort_values(by='Importância', ascending=False)

print("=== Top 5 Variáveis Mais Impactantes ===")
print(importancias.head(5))

=== Resultados da Avaliação ===
MAE (Erro Médio Absoluto): 1.25 dias
RMSE (Raiz do Erro Quadrático Médio): 1.57 dias
R² (Score de Variância): -0.08

=== Top 5 Variáveis Mais Impactantes ===
              Variável  Importância
3        freight_value     0.339883
0          order_value     0.325726
2   delivery_time_days     0.144702
1       items_quantity     0.090473
7  customer_region_Sul     0.025779


In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Carregar os dados
df = pd.read_csv('../data/desafio_nps_fase_1.csv')

# 2. Limpeza Cirúrgica (Prevenção de Data Leakage)
colunas_para_remover = [
    'customer_id', 'order_id',                  # Identificadores
    'nps_score', 'csat_internal_score',         # Pesquisas de satisfação
    'customer_service_contacts', 'complaints_count', 'resolution_time_days', # Pós-venda
    'delivery_time_days', 'delivery_attempts',  # Ocorrências logísticas futuras
    'repeat_purchase_30d',                      # Ação futura do cliente
    'payment_installments'                      # Solicitado na regra de negócio
]

# Garantir que apenas colunas que existem no dataframe sejam removidas
colunas_presentes = [col for col in colunas_para_remover if col in df.columns]
df_limpo = df.drop(columns=colunas_presentes)

# 3. Tratamento de Variáveis Categóricas (Região)
df_limpo = pd.get_dummies(df_limpo, columns=['customer_region'], drop_first=True)

# 4. Separar Preditoras (X) e Alvo (y)
X = df_limpo.drop(columns=['delivery_delay_days'])
y = df_limpo['delivery_delay_days']

# 5. Dividir os dados em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Treinar o Modelo
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train, y_train)

# 7. Avaliar a performance realista
y_pred = modelo_rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=== Performance Realista do Modelo (Pré-Checkout) ===")
print(f"MAE (Erro Médio Absoluto): {mae:.2f} dias")
print(f"R² (Score de Variância): {r2:.2f}\n")

# 8. Variáveis que mais indicam risco de atraso no momento da compra
importancias = pd.DataFrame({
    'Variável': X_train.columns, 
    'Importância': modelo_rf.feature_importances_
}).sort_values(by='Importância', ascending=False)

print("=== Principais Fatores de Atraso Visíveis no Checkout ===")
print(importancias.head(5))


from sklearn.model_selection import cross_val_score

# Usaremos K-Fold com 5 divisões (cv=5)
# O "neg_mean_absolute_error" é porque o scikit-learn maximiza scores, 
# então ele inverte o sinal do erro (deixa negativo)
scores = cross_val_score(modelo_rf, X, y, cv=5, scoring='neg_mean_absolute_error')

# Transformamos os erros negativos em positivos
mae_scores = -scores

print(f"MAEs para cada fold: {mae_scores}")
print(f"Média do MAE (Cross-Val): {mae_scores.mean():.2f}")
print(f"Desvio Padrão do MAE: {mae_scores.std():.2f}")

=== Performance Realista do Modelo (Pré-Checkout) ===
MAE (Erro Médio Absoluto): 1.25 dias
R² (Score de Variância): -0.07

=== Principais Fatores de Atraso Visíveis no Checkout ===
                 Variável  Importância
4          discount_value     0.198973
5           freight_value     0.191973
2             order_value     0.181195
1  customer_tenure_months     0.164215
0            customer_age     0.132314
MAEs para cada fold: [1.19006 1.15552 1.19992 1.26348 1.21308]
Média do MAE (Cross-Val): 1.20
Desvio Padrão do MAE: 0.04
